## 15. Loss Given Default (LGD) Modeling

In [ ]:
print("Filtering to defaulted loans...")
defaulted_df = df[df['loan_status'].isin(['Charged Off', 'Default'])].copy()
print(f"Defaulted loans: {len(defaulted_df):,} ({len(defaulted_df)/len(df):.2%} of portfolio)")

# Basic LGD: (loan_amnt - total_pymnt) / loan_amnt
defaulted_df['lgd'] = (
    (defaulted_df['loan_amnt'] - defaulted_df['total_pymnt']) / defaulted_df['loan_amnt']
).clip(0, 1)

print(f"Mean LGD:   {defaulted_df['lgd'].mean():.4f}")
print(f"Median LGD: {defaulted_df['lgd'].median():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(defaulted_df['lgd'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].axvline(defaulted_df['lgd'].mean(), color='black', linestyle='--',
                label=f"Mean: {defaulted_df['lgd'].mean():.3f}")
axes[0].set_title('LGD Distribution — Defaulted Loans'); axes[0].legend()
sns.kdeplot(defaulted_df['lgd'], ax=axes[1], color='#e74c3c', fill=True, alpha=0.4)
axes[1].set_title('LGD Density Plot')
plt.tight_layout(); plt.show()

# By grade
grade_lgd = defaulted_df.groupby('grade')['lgd'].mean().sort_index()
grade_lgd.plot(kind='bar', color='#e74c3c', alpha=0.8, title='Mean LGD by Grade')
plt.tight_layout(); plt.show()

In [ ]:
# Refined LGD: also includes post-default collections, net of collection fees
defaulted_df['total_recovered_refined'] = (
    defaulted_df['total_pymnt'] + defaulted_df['recoveries'] - defaulted_df['collection_recovery_fee']
)
defaulted_df['lgd_refined'] = (
    (defaulted_df['loan_amnt'] - defaulted_df['total_recovered_refined']) / defaulted_df['loan_amnt']
).clip(0, 1)

print(f"LGD (basic):   {defaulted_df['lgd'].mean():.4f}")
print(f"LGD (refined): {defaulted_df['lgd_refined'].mean():.4f}")

defaulted_df['lgd'] = defaulted_df['lgd_refined']  # use refined going forward
print(f"Using refined LGD: mean={defaulted_df['lgd'].mean():.4f}")

**LGD basic = 0.4672** — only payments made while loan was active. **LGD refined = 0.4137** — also includes post-default collections minus collection fees.

Lending Club LGD is NOT the typical bimodal spike-at-1.0 seen in mortgage data. Left spike near 0 and broad right hump peaking ~0.6 — personal loans have no collateral but borrowers typically make partial payments before defaulting.

### Two-Stage LGD Model

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

defaulted_df['lgd'] = defaulted_df['lgd_refined']

lgd_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt',
    'annual_inc', 'revol_util', 'loan_to_income',
    'mort_acc', 'has_pub_rec', 'grade_woe', 'home_ownership_woe'
]

lgd_df = defaulted_df[['loan_amnt', 'lgd', 'grade', 'purpose']].copy()
for feat in lgd_features:
    if feat in defaulted_df.columns: lgd_df[feat] = defaulted_df[feat]
    elif feat in df_cleaned.columns: lgd_df[feat] = df_cleaned[feat].reindex(defaulted_df.index)
lgd_df = lgd_df.dropna(subset=lgd_features + ['lgd'])
lgd_df['stage1_target'] = (lgd_df['lgd'] >= 0.10).astype(int)

X_lgd = lgd_df[lgd_features]
y1 = lgd_df['stage1_target']
y_lgd_vals = lgd_df['lgd']

X_tr, X_te, y1_tr, y1_te, yl_tr, yl_te = train_test_split(
    X_lgd, y1, y_lgd_vals, test_size=0.2, random_state=42, stratify=y1)

scaler_lgd = StandardScaler()
X_tr_s = scaler_lgd.fit_transform(X_tr)
X_te_s = scaler_lgd.transform(X_te)

# Stage 1 — logistic
s1 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
s1.fit(X_tr_s, y1_tr)
print(f"Stage 1 AUC: {roc_auc_score(y1_te, s1.predict_proba(X_te_s)[:,1]):.4f}")

# Stage 2 — linear on loss group
loss_mask = y1_tr == 1
s2 = LinearRegression()
s2.fit(X_tr_s[loss_mask], yl_tr[loss_mask])

def predict_lgd(X_input):
    Xs = scaler_lgd.transform(X_input)
    prob = s1.predict_proba(Xs)[:,1]
    lgd_loss = np.clip(s2.predict(Xs), 0, 1)
    return prob * lgd_loss + (1 - prob) * 0.05

preds_v1 = predict_lgd(X_te)
rmse_v1 = np.sqrt(mean_squared_error(yl_te, preds_v1))
print(f"Two-Stage LGD V1 RMSE: {rmse_v1:.4f}")
print(f"Mean Predicted: {preds_v1.mean():.4f} | Mean Actual: {yl_te.mean():.4f}")

In [ ]:
# V2: add behavioral features (loan age at default, payment ratio)
defaulted_df['loan_age_at_default'] = survival_df['duration_months'].reindex(defaulted_df.index)
defaulted_df['payment_ratio'] = (defaulted_df['total_pymnt'] / defaulted_df['loan_amnt']).clip(0, 1)
if 'installment' in defaulted_df.columns:
    defaulted_df['months_paid_ratio'] = (defaulted_df['total_pymnt'] / defaulted_df['installment']).clip(0, None)
defaulted_df['recoveries_ratio'] = (defaulted_df['recoveries'] / defaulted_df['loan_amnt']).clip(0, 1)

print("--- Correlation with LGD ---")
for feat in ['loan_age_at_default', 'payment_ratio', 'recoveries_ratio', 'int_rate', 'grade_woe']:
    if feat in defaulted_df.columns:
        corr = defaulted_df[[feat,'lgd']].dropna().corr().iloc[0,1]
        print(f"  {feat:<30} {corr:>8.4f}")

lgd_features_v2 = lgd_features + ['loan_age_at_default', 'payment_ratio', 'recoveries_ratio']
if 'months_paid_ratio' in defaulted_df.columns:
    lgd_features_v2.append('months_paid_ratio')

lgd_df_v2 = defaulted_df[['loan_amnt','lgd','grade']].copy()
for feat in lgd_features_v2:
    if feat in defaulted_df.columns: lgd_df_v2[feat] = defaulted_df[feat]
    elif feat in df_cleaned.columns: lgd_df_v2[feat] = df_cleaned[feat].reindex(defaulted_df.index)
lgd_df_v2 = lgd_df_v2.dropna(subset=lgd_features_v2+['lgd'])
lgd_df_v2['stage1_target'] = (lgd_df_v2['lgd'] >= 0.10).astype(int)

X2 = lgd_df_v2[lgd_features_v2]
y1_2 = lgd_df_v2['stage1_target']
yl2 = lgd_df_v2['lgd']

X2_tr, X2_te, y1_2_tr, y1_2_te, yl2_tr, yl2_te = train_test_split(
    X2, y1_2, yl2, test_size=0.2, random_state=42, stratify=y1_2)

scaler_v2 = StandardScaler()
X2_tr_s = scaler_v2.fit_transform(X2_tr)
X2_te_s = scaler_v2.transform(X2_te)

s1_v2 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
s1_v2.fit(X2_tr_s, y1_2_tr)

lm_v2 = y1_2_tr == 1
s2_v2 = LinearRegression()
s2_v2.fit(X2_tr_s[lm_v2], yl2_tr[lm_v2])

def predict_lgd_v2(X_input):
    Xs = scaler_v2.transform(X_input)
    prob = s1_v2.predict_proba(Xs)[:,1]
    lgd_loss = np.clip(s2_v2.predict(Xs), 0, 1)
    return prob * lgd_loss + (1 - prob) * 0.05

preds_v2 = predict_lgd_v2(X2_te)
PORTFOLIO_LGD = preds_v2.mean()
rmse_v2 = np.sqrt(mean_squared_error(yl2_te, preds_v2))
print(f"\nTwo-Stage LGD V2 RMSE: {rmse_v2:.4f}")
print(f"Portfolio LGD: {PORTFOLIO_LGD:.4f}")

# Grade comparison
res_v2 = X2_te.copy()
res_v2['actual'] = yl2_te.values
res_v2['predicted'] = preds_v2
res_v2['grade'] = lgd_df_v2.loc[X2_te.index, 'grade']
print("\n--- LGD by Grade: Actual vs Predicted ---")
print(res_v2.groupby('grade').agg(actual=('actual','mean'), predicted=('predicted','mean'), count=('actual','count')).round(4))

**Result:** Two-stage LGD V2 achieves RMSE=0.019 after adding behavioral features. Payment ratio (correlation -0.974 with LGD) and loan age at default (-0.796) dominate — LGD for unsecured loans is driven by borrower payment behavior, not origination characteristics. Grade-level predictions within 0.006 of actuals. Portfolio LGD = 0.41.